In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("features_20d.csv", index_col=0, parse_dates=True)

print(data.shape)
print(data.head())

(7917, 8)
            rolling_mean  realized_vol  skewness  kurtosis  vix_level  \
Date                                                                    
1993-07-22      0.000149      0.091949  0.360993 -0.168377      11.69   
1993-07-23      0.000054      0.090106  0.361060 -0.026061      11.32   
1993-07-26      0.000166      0.091161  0.309674 -0.170904      11.32   
1993-07-27     -0.000401      0.084488  0.428593  0.438424      11.34   
1993-07-28     -0.000390      0.084415  0.424212  0.446069      11.37   

            vix_change  vix_spread  corr_sp500_vix  
Date                                                
1993-07-22   -0.023392    0.024951       -0.056248  
1993-07-23   -0.031651    0.023094       -0.136878  
1993-07-26    0.000000    0.022039       -0.137696  
1993-07-27    0.001767    0.028912       -0.102200  
1993-07-28    0.002645    0.029285       -0.104895  


In [2]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)

scaled_df = pd.DataFrame(
    scaled_data,
    index=data.index,
    columns=data.columns
)

In [3]:
def create_sequences(data, seq_len=30):
    X = []
    
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        
    return np.array(X)

SEQ_LEN = 30

X = create_sequences(scaled_df.values, SEQ_LEN)

print("LSTM input shape:", X.shape)

LSTM input shape: (7887, 30, 8)


In [4]:
train_size = int(0.8 * len(X))

X_train = X[:train_size]
X_test  = X[train_size:]

print(X_train.shape, X_test.shape)

(6309, 30, 8) (1578, 30, 8)


In [5]:
import torch
from torch.utils.data import DataLoader, TensorDataset

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_tensor),
    batch_size=32,
    shuffle=True
)

In [6]:
import torch.nn as nn

class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, latent_dim=16):
        super().__init__()
        
        # Encoder
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=latent_dim,
            batch_first=True
        )
        
        # Decoder
        self.decoder = nn.LSTM(
            input_size=latent_dim,
            hidden_size=n_features,
            batch_first=True
        )
        
    def forward(self, x):
        # Encode
        _, (hidden, _) = self.encoder(x)
        
        # Repeat latent vector
        latent = hidden.repeat(x.shape[1], 1, 1).permute(1, 0, 2)
        
        # Decode
        out, _ = self.decoder(latent)
        
        return out

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMAutoencoder(
    n_features=X.shape[2],
    latent_dim=16
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [8]:
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        x = batch[0].to(device)
        
        optimizer.zero_grad()
        
        output = model(x)
        loss = criterion(output, x)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

Epoch 1/20, Loss: 157.6801
Epoch 2/20, Loss: 132.7363
Epoch 3/20, Loss: 124.0480
Epoch 4/20, Loss: 118.4689
Epoch 5/20, Loss: 114.3091
Epoch 6/20, Loss: 111.5573
Epoch 7/20, Loss: 109.7459
Epoch 8/20, Loss: 108.0020
Epoch 9/20, Loss: 106.1660
Epoch 10/20, Loss: 105.3946
Epoch 11/20, Loss: 103.8715
Epoch 12/20, Loss: 103.6923
Epoch 13/20, Loss: 102.9839
Epoch 14/20, Loss: 102.5234
Epoch 15/20, Loss: 102.2512
Epoch 16/20, Loss: 103.4343
Epoch 17/20, Loss: 101.8958
Epoch 18/20, Loss: 101.4710
Epoch 19/20, Loss: 101.5020
Epoch 20/20, Loss: 101.1240


In [9]:
model.eval()

LSTMAutoencoder(
  (encoder): LSTM(8, 16, batch_first=True)
  (decoder): LSTM(16, 8, batch_first=True)
)

In [13]:
def get_embeddings(model, X):
    model.eval()
    embeddings = []
    
    with torch.no_grad():
        for i in range(len(X)):
            x = torch.tensor(X[i:i+1], dtype=torch.float32).to(device)
            
            _, (hidden, _) = model.encoder(x)
            
            # FIX: remove all extra dimensions
            z = hidden.squeeze().cpu().numpy()   # <-- IMPORTANT FIX
            
            embeddings.append(z)
    
    return np.array(embeddings)

In [14]:
embeddings = get_embeddings(model, X)

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (7887, 16)


In [15]:
embedding_df = pd.DataFrame(
    embeddings,
    index=data.index[SEQ_LEN:],
)

print(embedding_df.head())

                  0         1         2         3         4         5   \
Date                                                                     
1993-09-02  0.760285 -0.924129 -0.009925  0.034997 -0.120778 -0.102178   
1993-09-03  0.757770 -0.926769 -0.012095  0.032252 -0.129407 -0.097624   
1993-09-07  0.774869 -0.927917 -0.018949  0.014952 -0.123583 -0.107257   
1993-09-08  0.759272 -0.924297 -0.003885  0.023663 -0.138683 -0.073905   
1993-09-09  0.772817 -0.920981  0.020676  0.024855 -0.131285 -0.063255   

                  6         7         8         9         10        11  \
Date                                                                     
1993-09-02  0.681038 -0.633212 -0.142053  0.315817 -0.760834 -0.306202   
1993-09-03  0.683469 -0.641021 -0.145165  0.317915 -0.770288 -0.295173   
1993-09-07  0.698873 -0.646146 -0.158129  0.316817 -0.775543 -0.319128   
1993-09-08  0.643912 -0.645471 -0.139675  0.307829 -0.768308 -0.277567   
1993-09-09  0.630517 -0.647470 -0.114

In [16]:
embedding_df.to_csv("lstm_embeddings.csv")
print("Saved embeddings")

Saved embeddings
